# Automated ML Project Architect

In [31]:
# Importing Necessary Libraries:

import os
from dotenv import load_dotenv
import pandas as pd
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
openai = OpenAI()

## Step-1: Pandas Extractor

In [2]:
data = pd.read_csv('loan_borowwer_data.csv')
data.head()

,credit.policy,purpose,int.rate,installment,log.annual.inc,dti,fico,days.with.cr.line,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec,not.fully.paid
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,0
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,0
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,0
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,0
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,0


In [14]:
# Function to Load in CSV and Getting Columns, Data-Types and First Few Rows:

def data_profile(file_path):

    # Loading Given CSV File:
    df = pd.read_csv(file_path)

    # Getting Column Names:
    column_names = df.columns.to_list()
    #print(column_names)

    # Getting Data Types:
    data_types = df.dtypes.to_list()
    #print(data_types)

    # Getting First few Rows for Example Data:
    data_sample = df.head().to_string(header= False)
    #print(data_sample)

    # Getting Missing value Counts:
    missing_count = df.isnull().sum().to_string()
    #print(missing_count)

    # Getting Summary Statistics:
    summary_stats = df.describe().to_string()
    #print(summary_stats)

    return column_names, data_types, data_sample, missing_count, summary_stats

In [15]:
# Testing Above function:
a,b,c,d,e = data_profile(file_path= 'C://Users//shail//PycharmProjects//PythonProject//Applied-LLM-Engineering//01_Exploring_Top_Models//loan_borowwer_data.csv')

In [16]:
print(a, b, c, d, e)

['credit.policy', 'purpose', 'int.rate', 'installment', 'log.annual.inc', 'dti', 'fico', 'days.with.cr.line', 'revol.bal', 'revol.util', 'inq.last.6mths', 'delinq.2yrs', 'pub.rec', 'not.fully.paid'] [dtype('int64'), dtype('O'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('int64'), dtype('float64'), dtype('int64'), dtype('float64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64')] 0  1  debt_consolidation  0.1189  829.10  11.350407  19.48  737  5639.958333  28854  52.1  0  0  0  0
1  1         credit_card  0.1071  228.22  11.082143  14.29  707  2760.000000  33623  76.7  0  0  0  0
2  1  debt_consolidation  0.1357  366.86  10.373491  11.63  682  4710.000000   3511  25.6  1  0  0  0
3  1  debt_consolidation  0.1008  162.34  11.350407   8.10  712  2699.958333  33667  73.2  1  0  0  0
4  1         credit_card  0.1426  102.92  11.299732  14.97  667  4066.000000   4740  39.5  0  1  0  0 credit.policy        0
purpose              0
int.rate    

In [32]:
# Writing a Function for API Call to LLM:

def pandas_extractor(column_names, data_types, data_sample, missing_count, summary_stats):

    # Defining System prompt:
    system_prompt = """
    You are a Senior Data Scientist. Your Job is to analyse the given Data Profile and help the user with deciding Data Cleaning Process and Problem Type and Model Recommendations.

    Output a JSON object containing the target_variable, problem_type, recommended_models, and specific data_cleaning_warnings.

    Output Should be in this Specific JSON Format:
    {
    "target_variable": "Churn",
    "problem_type": "Binary Classification",
    "recommended_models": ["Logistic Regression", "Support Vector Machine"],
    "data_cleaning_warnings": ["'Date' column is a string", "'Age' has missing values"]
    }
    """

    # Defining User prompt:
    user_prompt = f"""
    Here are the column names:
    {column_names}

    Here are the data types:
    {data_types}

    Here is a sample of the data:
    {data_sample}

    Here is the count of missing values per column:
    {missing_count}

    Here are the summary statistics:
    {summary_stats}

    Based on this complete data profile, please help me decide data cleaning steps, problem type and model recommendations.

    """

    # Defining Message:
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]

    # Calling LLM API:
    response = openai.chat.completions.create(model= 'gpt-5-mini',
                                              messages= messages,
                                              response_format= {'type': 'json_object'})

    return response.choices[0].message.content


In [33]:
# Testing pandas_extractor Function:
column_names, data_types, data_sample, missing_count, summary_stats = data_profile('loan_borowwer_data.csv')

test_res = pandas_extractor(column_names= column_names,
                            data_types= data_types,
                            data_sample= data_sample,
                            missing_count= missing_count,
                            summary_stats= summary_stats)

In [34]:
print(test_res)

{
  "target_variable": "not.fully.paid",
  "problem_type": "Binary Classification",
  "recommended_models": [
    "Logistic Regression (with class_weight or regularization)",
    "Random Forest",
    "Gradient Boosting (XGBoost or LightGBM)",
    "CatBoost (handles categorical 'purpose' well)",
    "Support Vector Machine (with scaling) — for smaller experiments"
  ],
  "data_cleaning_warnings": [
    "'purpose' is object/categorical — requires encoding (one-hot, target/frequency encoding, or use CatBoost's native handling)",
    "'revol.bal' is highly skewed with a very large max (~1.2e6) — check for outliers, consider capping or log-transform",
    "'installment' and 'revol.bal' distributions are skewed — consider transformation or robust scaling for some models",
    "'log.annual.inc' is already log-transformed — do not re-log; consider exponentiating only for interpretability if needed",
    "'int.rate' is expressed as a decimal (e.g. 0.1189 = 11.89%) — ensure consistent interpreta

## Step-2: The Tech Writer

In [38]:
# Defining a Function to Write A Project Markdown File Using LLM API Call:

def tech_writer(json_summary):

    system_prompt = """
    You are to Act as an Senior Tech Writer. Your job is to analyze given JSON Summary of Data and Recommendations and Strategy by Senior Data Scientist and write A Highly Professional, Step by Step, formatted README.md file for a GitHub Repository.

    CRITICAL INSTRUCTIONS:
    - You must output ONLY the raw Markdown text.
    - Do not include any conversational filler, greetings, or sign-offs.
    - Do NOT offer follow-up assistance, suggestions, or ask what the user wants to do next.
    - The very last character of your output must be the end of the README document.
    """

    user_prompt = f"""
    Here is Recommendations and Strategy Given by Senior Data Scientist: {json_summary}.
    Please create a Professional Markdown Document explaining the business problem, why those specific models were chosen, and what data cleaning steps need to happen first.
    """

    # Defining Message:
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]

    # Defining Call to LLM API:
    response = openai.chat.completions.create(model= 'gpt-5-mini',
                                              messages= messages)

    return response.choices[0].message.content


In [39]:
# Testing Above Function:
column_names, data_types, data_sample, missing_count, summary_stats = data_profile('loan_borowwer_data.csv')

test_res = pandas_extractor(column_names= column_names,
                            data_types= data_types,
                            data_sample= data_sample,
                            missing_count= missing_count,
                            summary_stats= summary_stats)

markdown_doc = tech_writer(json_summary=test_res)

display(Markdown(markdown_doc))

# README

## Credit Default Prediction — Project Overview

This repository contains an end-to-end plan for building a binary classification model to predict loan default, where the target variable is `not.fully.paid`. The goal is to identify borrowers at elevated risk of not fully repaying their loans so that downstream teams (underwriting, risk, collections) can act on predictions with appropriate business rules.

Problem type: Binary Classification  
Target positive class: "not fully paid" (~16% prevalence)

---

## Business Problem Statement

- Objective: Accurately identify loans that are likely to be not fully paid to reduce credit losses and optimize lending decisions.
- Impact metrics: reduction in charge-offs, improved risk-adjusted yield, cost of false positives vs false negatives (business-defined).
- Constraints & considerations: imbalanced target (~16% positives), potential data quality issues, and the importance of avoiding target leakage from post-loan information.

---

## Target Variable

- name: `not.fully.paid`
- interpretation: binary flag indicating whether the loan was not fully repaid.
- prevalence: ≈16% positives (class imbalance must be addressed).

---

## Recommended Modeling Approaches and Why

1. Logistic Regression (regularized, with class weights)
   - Strong baseline: interpretable coefficients, fast training and inference.
   - Regularization reduces overfitting and addresses multicollinearity.
   - Class weights penalize misclassification of minority class without resampling.

2. Random Forest
   - Robust to outliers and skewed features, handles non-linearities and interactions.
   - Requires limited feature scaling.
   - Good baseline for feature importance and stability.

3. Gradient Boosting (XGBoost / LightGBM / CatBoost)
   - State-of-the-art performance on tabular data.
   - Handles categorical encoding differently (CatBoost natively supports categories).
   - Good handling of class imbalance through objective weighting and sampling.

4. Support Vector Machine (with feature scaling)
   - Effective in high-dimensional spaces and with margin maximization.
   - Requires careful scaling; suitable for smaller training sets or strong regularization.

5. Neural Network (MLP)
   - Flexible non-linear model; can capture complex interactions.
   - Requires substantial preprocessing (scaling, class weighting, regularization) and careful tuning.
   - Most useful when sufficient data and compute are available; benefits from early stopping and dropout.

Selection rationale:
- Use a mix of interpretable (Logistic) and high-performance non-linear models (RF, GBDT, MLP) to balance explainability and predictive power.
- Include SVM for a complementary margin-based classifier where appropriate.
- CatBoost is recommended when leveraging categorical features directly without heavy encoding.

---

## Primary Evaluation Metrics

- AUC-ROC: global ranking performance.
- Precision-Recall (PR) curve and average precision: sensitive to class imbalance; important for the minority class.
- Precision@k or recall@k (business-specific thresholds).
- Confusion-matrix-derived metrics (precision, recall, F1) at selected decision thresholds.
- Calibration metrics (Brier score, calibration plots) — probability outputs should be well calibrated for risk-based decisions.
- Use cost-sensitive metrics if business cost of false positives/negatives is defined.

---

## Data Cleaning & Preprocessing — Required First Steps (Step-by-step)

Follow these steps in order. Treat each step as a reproducible, testable operation implemented in a data pipeline/pipeline stage with logging.

1. Initial Ingestion & Schema Validation
   - Load raw data and assert column presence and types.
   - Save a copy of raw data (immutable) and document ingestion timestamp and source.
   - Verify data file encoding and parsing rules (delimiters, quote chars).

2. Sanity Checks & Summary Statistics
   - Compute per-column counts, unique counts, min/max/mean/std, and missing-value counts.
   - Confirm overall row counts match expected sample sizes.
   - Record feature dtype inference and override where necessary.

3. Verify Missing Values
   - Even though the current profile reports no missing values, re-check after parsing/merging.
   - Log any missingness patterns (per-feature, per-row) and decide imputation strategy if needed.

4. Fix Incorrect dtypes and Flags
   - Convert features that are numeric but represent categories/flags to categorical dtype (e.g., `credit.policy` if it is a flag).
   - Ensure target `not.fully.paid` is binary (0/1) and properly typed.

5. Categorical Feature: `purpose`
   - `purpose` is object/categorical. Options:
     - One-hot encoding (if low cardinality).
     - Target or ordinal encoding (if many categories and leakage risk is controlled).
     - Use CatBoost/LightGBM/Tree models that accept categorical features directly (CatBoost preferred for native handling).
   - When using target encoding, implement cross-fold encoding to prevent leakage.

6. `log.annual.inc` Provenance
   - Document and assert that `log.annual.inc` is already log-transformed. Do NOT re-transform.
   - Record original annual income source if available.

7. Validate Units & Ranges
   - `revol.util` has values > 100 (max 119): confirm units (percent vs ratio). If percent, cap or correct obvious data errors (e.g., values >100%).
   - `days.with.cr.line` contains fractional values and a large range: confirm what unit `days.with.cr.line` represents (days vs months/years) and convert to a consistent unit. If fractional values are logically inconsistent, verify ingestion precision.

8. Handle Skewed & Heavy-Tailed Features
   - `revol.bal` and `installment` have very large maxima and heavy right skew.
     - Consider log(x + c) transformation, winsorization (e.g., 1st/99th percentiles), or robust scaling.
     - Keep original distributions documented for interpretability and reproducibility.
   - Integer-count features (`inq.last.6mths`, `delinq.2yrs`, `pub.rec`) are heavily skewed.
     - Consider binning into ordinal categories (0, 1, 2-4, 5+) or leave as counts and use models robust to skew (tree-based).
     - If modeled as continuous, use robust scaling or transformations.

9. Outlier Detection & Treatment
   - Identify extreme outliers using robust methods (IQR, median absolute deviation) and domain rules.
   - Decide per-feature whether to:
     - Cap/winsorize, remove, or keep (for tree models).
     - Document all changes and retain original values in raw data.

10. Multicollinearity Assessment
    - Compute pairwise correlations and Variance Inflation Factor (VIF) for continuous features.
    - Pay attention to potential collinearity between `log.annual.inc`, `installment`, and `dti`.
    - Strategies: remove or combine correlated predictors, use regularization (L1/L2), or use tree-based models that handle correlation.

11. Target Leakage Audit
    - Manually inspect each feature to ensure no information from future/post-loan events is included.
    - Remove or transform any features derived or collected after loan outcome.

12. Train / Validation / Test Splits
    - Use stratified sampling on the target to preserve class imbalance across splits.
    - For temporal data: use time-based split (train earlier loans, test later loans) to mimic production.
    - Maintain a holdout test set not used during model selection/tuning.

13. Scaling & Feature Pipelines
    - For distance-based or regularized models (SVM, Logistic Regression, Neural Nets):
      - Apply StandardScaler or RobustScaler (robust to outliers).
    - For tree-based models: scaling is not required.
    - Build scikit-learn pipelines (or equivalent) to encapsulate imputation, encoding, scaling, and modeling to ensure transformations are applied identically in training and inference.

14. Imbalanced Target Handling
    - Options (experiment and validate impact):
      - Class weighting (preferred for logistic & many classifiers): set class_weight or weight samples.
      - Resampling:
        - Oversampling minority: SMOTE (ensure SMOTE applied within cross-validation folds).
        - Undersampling majority (random or informed).
        - Hybrid methods (SMOTE + Tomek links).
      - Threshold tuning and cost-sensitive decision rules post-probability calibration.
    - Avoid naively applying resampling prior to train/test split; apply resampling inside training folds only.

15. Cross-Validation & Hyperparameter Tuning
    - Use stratified K-fold CV (or time-series CV if temporal).
    - Tune hyperparameters for regularization, tree depth/estimators/learning rate, SVM C/gamma, MLP architecture/lr/regularization.
    - Use nested CV or holdout validation for final selection.

16. Probability Calibration
    - Calibrate probability outputs (Platt scaling, isotonic regression) when probability estimates are used for business decisions.

17. Model Explainability & Monitoring
    - Use SHAP or permutation feature importance for model explanation.
    - Implement monitoring for data drift, feature distributions, and model performance decay.
    - Log model artifacts, feature schemas, and training metadata.

---

## Suggested Feature Engineering Ideas

- Interaction terms between `dti`, `installment`, and `log.annual.inc`.
- Ratio features: installment / monthly_income (derive monthly_income from `log.annual.inc` if provenance known).
- Binning of counts (credit inquiries, delinquencies).
- Temporal features derived from `days.with.cr.line` (age of credit line in years/months).
- Aggregated categorical group statistics (e.g., average default rate per purpose) added with proper cross-validation to avoid leakage.

---

## Model-specific Preprocessing Checklist

- Logistic Regression
  - One-hot or appropriately encoded categories.
  - Standardize all continuous features (StandardScaler / RobustScaler).
  - Use regularization (L1/L2) and class_weight.

- SVM
  - Robust scaling is recommended.
  - Consider dimensionality reduction (PCA) if many features.

- Neural Network (MLP)
  - Standardize inputs.
  - Use dropout, early stopping, and class weights or focal loss.
  - Normalize target sampling rate in minibatches if extreme imbalance.

- Random Forest / Gradient Boosting
  - Minimal scaling required.
  - Prefer LightGBM/CatBoost when many categorical variables or for speed.
  - Use appropriate objective & class weighting or balanced sampling.

---

## Validation Strategy

- Primary: Stratified K-fold CV (k = 5 or 10) to evaluate model stability.
- For time-dependent datasets: Use time-ordered validation (rolling origin) to avoid lookahead bias.
- Keep a final untouched holdout set for final model evaluation.
- Report mean and standard deviation of evaluation metrics across folds.

---

## Reproducibility, Deployment & Logging

- Fix random seeds for data splits, model initialization, and resampling.
- Store model artifacts, preprocessing pipelines, and metadata (feature list, feature types, scaler parameters).
- Version datasets and code (e.g., DVC or dataset snapshot).
- Maintain a model card documenting dataset, intended use, performance, limitations, and biases.

---

## Quick Starter Checklist (First Run)

1. Ingest raw dataset and assert schema.
2. Run schema validation and missing-value checks.
3. Convert dtypes and encode flags (e.g., `credit.policy`).
4. Validate `revol.util` units and fix >100 values.
5. Do not re-transform `log.annual.inc`; document.
6. Treat skewed features (winsorize/log/robust scale) or leave for tree models.
7. Encode `purpose` appropriately (or use CatBoost).
8. Stratified train/validation/test split (or time-based split).
9. Build scikit-learn pipeline including encoding, scaling, and classifier.
10. Train baseline Logistic Regression with class weights and evaluate AUC and PR-AUC.

---

## Implementation Roadmap (Suggested Milestones)

- Milestone 0: Environment, data ingestion, and schema validation.
- Milestone 1: Baseline preprocessing + Logistic Regression baseline with class weights.
- Milestone 2: Tree-based models (Random Forest, XGBoost/LightGBM/CatBoost) and hyperparameter tuning.
- Milestone 3: Advanced resampling experiments (SMOTE vs class weights), SVM and MLP experiments.
- Milestone 4: Model calibration, interpretability (SHAP), and final model selection.
- Milestone 5: Productionization: pipeline hardening, monitoring, and model deployment.

---

## Important Notes & Risks

- Always check for target leakage before modeling — leakage can produce deceptively good results that fail in production.
- Document all transformations and keep raw data immutable for traceability.
- Treat units and provenance (e.g., `log.annual.inc`, `days.with.cr.line`) carefully; incorrect assumptions can invalidate models.
- Evaluate models under business-specific cost functions when available — choosing a metric only on AUC may not reflect operational impact.

---

## Files & Artifacts (Recommended Repository Layout)

- data/
  - raw/ (immutable)
  - processed/
- src/
  - ingestion.py
  - validate_schema.py
  - preprocessing.py
  - features.py
  - train.py
  - evaluate.py
  - explainability.py
- notebooks/
  - eda.ipynb
  - modeling_baselines.ipynb
- models/
  - checkpoints/
  - final/
- tests/
  - test_schema.py
  - test_preprocessing.py
- README.md (this file)

---

End of README.